# Pakistan News Intelligence Research
### Automated NLP Analysis of 350,000 Headlines from Dawn News
This notebook presents a visual analysis of news data focused on Pakistan's domestic policy, regional security, and socio-economic trends.

In [9]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os
from collections import Counter

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

df = pd.read_parquet('data/processed/processed_nlp_features.parquet')
df['published_at'] = pd.to_datetime(df['published_at'])

print(f"Successfully loaded {len(df):,} Pakistan-centric headlines.")
display(df.head(3))

Successfully loaded 5,000 Pakistan-centric headlines.


,id,published_at,headline,category,body,url,extracted_entities,sentiment_score,sentiment_label,dominant_topic
0,163749,2017-12-04,Several PPP political figures join PTI,Pakistan,The former PPP members called on PTI leader an...,https://www.dawn.com/news/1374381/several-ppp-...,"[PTI (ORG), PPP (ORG)]",0.2960,positive,7
1,151801,2017-06-10,"Woman accused of conspiring to kill nephew, hi...",Pakistan,Police claimed to have arrest the aunt and unc...,https://www.dawn.com/news/1338568/woman-accuse...,[],-0.7845,negative,2
2,125496,2016-04-26,Ahmed Shehzad hurls bat to break dressing room...,Sport,A PCB official said Ahmed has assured to bear ...,https://www.dawn.com/news/1254390/ahmed-shehza...,[Ahmed Shehzad (PERSON)],0.0000,neutral,9


## 1. Dataset Overview
A high-level summary of the dataset volume and the overall emotional baseline of Pakistan's media landscape calculated via VADER sentiment analysis.

In [10]:
total_headlines = len(df)
avg_sentiment = df['sentiment_score'].mean()
unique_topics = df['dominant_topic'].nunique()

print(f"Total Headlines: {total_headlines:,}")
print(f"National Avg Sentiment: {avg_sentiment:.3f}")
print(f"Total Unique Topics: {unique_topics}")

Total Headlines: 5,000
National Avg Sentiment: -0.093
Total Unique Topics: 10


## 2. Topic Distribution
This chart displays the volume of news within each numerical topic ID. It highlights the concentration of reporting across the modeled categories.

In [11]:
topic_data = df['dominant_topic'].value_counts().reset_index()
topic_data.columns = ['Topic_ID', 'Count']
topic_data['Topic_ID'] = topic_data['Topic_ID'].astype(str)

fig_topics = px.bar(
    topic_data, 
    x='Count', 
    y='Topic_ID', 
    orientation='h',
    title='News Volume by Topic ID',
    color='Count',
    color_continuous_scale='RdBu_r'
)
fig_topics.update_layout(yaxis={'categoryorder':'total ascending'}, height=600)
fig_topics.show()

## 3. Sentiment Pulse
A 7-day rolling average to identify major shifts in the national mood. Dips often correspond to periods of political instability, security incidents, or economic inflation data.

In [12]:
daily_series = df.groupby(df['published_at'].dt.date)['sentiment_score'].mean().reset_index()
daily_series['rolling_avg'] = daily_series['sentiment_score'].rolling(window=7).mean()

fig_trend = px.line(
    daily_series, 
    x='published_at', 
    y='rolling_avg',
    title='National News Sentiment Trend (7-Day Rolling)',
    labels={'rolling_avg': 'Sentiment Score', 'published_at': 'Date'}
)
fig_trend.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)
fig_trend.show()

## 4. Key Actors and Locations (NER)
Identifying the most prominent political figures, organizations, and cities mentioned in Dawn News headlines.

In [13]:
all_ents = [ent for sublist in df['extracted_entities'].dropna() for ent in sublist]
ent_counts = Counter(all_ents).most_common(15)
ent_df = pd.DataFrame(ent_counts, columns=['Entity', 'Mentions'])

fig_ents = px.pie(
    ent_df, 
    values='Mentions', 
    names='Entity', 
    title='Share of Voice: Prominent Entities in Pakistan News',
    hole=0.4
)
fig_ents.show()

## 5. Topic Specific Content Analysis
Examining specific samples from the dominant topics to interpret the numerical clusters and verify sentiment scoring.

In [14]:
target_id = df['dominant_topic'].unique()[0]
st_sample = df[df['dominant_topic'] == target_id].sort_values(by='sentiment_score')

print(f"Representative Headlines for Topic ID: {target_id}")
display(st_sample[['published_at', 'headline', 'sentiment_score']].head(3))
display(st_sample[['published_at', 'headline', 'sentiment_score']].tail(3))

Representative Headlines for Topic ID: 7


,published_at,headline,sentiment_score
2555,2020-04-13,"2-year-old killed, 4 injured in Indian fire of...",-0.9100
2446,2022-12-25,"Two suspects, shopkeeper killed as crime conti...",-0.8860
4286,2025-06-04,Suspect in Hafizabad double child murder case ...,-0.8779


,published_at,headline,sentiment_score
4937,2019-03-11,Zalmi at top of PSL points table after securin...,0.7845
1250,2016-06-02,European celebrities sign 'love letter' to Bri...,0.7906
3549,2021-05-25,"Asad Umar defends economic growth figures, cre...",0.7906
